# Extracción de Datos desde SQL Server

Este notebook demuestra cómo conectarse a SQL Server y extraer datos de la tabla `AdventureWorks.Person.Person` utilizando Python.

**Tecnologías utilizadas:**
- `pyodbc` — driver ODBC para conectarse a SQL Server
- `sqlalchemy` — ORM y motor de conexión
- `pandas` — manipulación y análisis de datos

## 1. Instalación de Dependencias

## 2. Importación de Librerías

In [1]:
import pyodbc
import pandas as pd
from sqlalchemy import create_engine, text
import urllib

print('Librerías importadas correctamente.')

Librerías importadas correctamente.


## 3. Parámetros de Conexión

> ⚠️ **Buena práctica:** En ambientes de producción, las credenciales deben almacenarse en variables de entorno o un vault de secretos, nunca en texto plano dentro del código.

In [2]:
# Parámetros de conexión
SERVIDOR  = r'localhost\SQLEXPRESS02'
BASE_DATOS = 'AdventureWorks'
USUARIO   = 'sergio'
CONTRASENA = '123456789'
DRIVER    = 'ODBC Driver 17 for SQL Server'  # Ajustar según el driver instalado

print(f'Servidor  : {SERVIDOR}')
print(f'Base datos: {BASE_DATOS}')
print(f'Usuario   : {USUARIO}')
print(f'Driver    : {DRIVER}')

Servidor  : localhost\SQLEXPRESS02
Base datos: AdventureWorks
Usuario   : sergio
Driver    : ODBC Driver 17 for SQL Server


## 4. Verificar Drivers ODBC Disponibles

In [8]:
# Listar los drivers ODBC instalados en el sistema
drivers = pyodbc.drivers()
print('Drivers ODBC disponibles:')
for d in drivers:
    print(f'  - {d}')

Drivers ODBC disponibles:
  - SQL Server
  - ODBC Driver 17 for SQL Server
  - ODBC Driver 18 for SQL Server


## 5. Conexión con pyodbc (conexión directa)

In [ ]:
# Cadena de conexión ODBC
conn_str = (
    f'DRIVER={{{DRIVER}}};'
    f'SERVER={SERVIDOR};'
    f'DATABASE={BASE_DATOS};'
    f'UID={USUARIO};'
    f'PWD={CONTRASENA};'
    'TrustServerCertificate=yes;'  # Necesario en SQL Server con certificado autofirmado
)

try:
    conn = pyodbc.connect(conn_str)
    print('✅ Conexión exitosa con pyodbc!')
except pyodbc.Error as e:
    print(f'❌ Error de conexión: {e}')
    conn = None

✅ Conexión exitosa con pyodbc!


## 6. Conexión con SQLAlchemy (recomendado para pandas)

In [ ]:
# Codificar la cadena de conexión para SQLAlchemy
params = urllib.parse.quote_plus(conn_str)
engine = create_engine(f'mssql+pyodbc:///?odbc_connect={params}')

try:
    with engine.connect() as conexion:
        resultado = conexion.execute(text('SELECT @@VERSION AS version'))
        version = resultado.fetchone()
        print('✅ Conexión exitosa con SQLAlchemy!')
        print(f'\nVersión de SQL Server:\n{version[0]}')
except Exception as e:
    print(f'❌ Error de conexión: {e}')

✅ Conexión exitosa con SQLAlchemy!

Versión de SQL Server:
Microsoft SQL Server 2025 (RTM-GDR) (KB5084814) - 17.0.1110.1 (X64) 
	Mar 13 2026 01:04:22 
	Copyright (C) 2025 Microsoft Corporation
	Express Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 26200: ) (Hypervisor)



## 7. Exploración de la Tabla Person.Person

In [ ]:
# Ver las columnas disponibles en Person.Person
query_columnas = """
SELECT
    COLUMN_NAME,
    DATA_TYPE,
    CHARACTER_MAXIMUM_LENGTH,
    IS_NULLABLE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'Person'
  AND TABLE_NAME   = 'Person'
ORDER BY ORDINAL_POSITION;
"""

df_columnas = pd.read_sql(query_columnas, engine)
print(f'Total de columnas: {len(df_columnas)}')
df_columnas

Total de columnas: 13


,COLUMN_NAME,DATA_TYPE,CHARACTER_MAXIMUM_LENGTH,IS_NULLABLE
0,BusinessEntityID,int,NaN,NO
1,PersonType,nchar,2.0,NO
2,NameStyle,bit,NaN,NO
3,Title,nvarchar,8.0,YES
4,FirstName,nvarchar,50.0,NO
5,MiddleName,nvarchar,50.0,YES
6,LastName,nvarchar,50.0,NO
7,Suffix,nvarchar,10.0,YES
8,EmailPromotion,int,NaN,NO
9,AdditionalContactInfo,xml,-1.0,YES


## 8. Extracción de Datos — Consulta Básica

In [ ]:
# Extraer las primeras 100 filas de Person.Person
query_personas = """
SELECT TOP 100
    BusinessEntityID,
    PersonType,
    NameStyle,
    Title,
    FirstName,
    MiddleName,
    LastName,
    Suffix,
    EmailPromotion,
    ModifiedDate
FROM Person.Person
ORDER BY BusinessEntityID;
"""

df_personas = pd.read_sql(query_personas, engine)

print(f'Filas extraídas : {len(df_personas)}')
print(f'Columnas        : {list(df_personas.columns)}')
df_personas.head(10)

Filas extraídas : 100
Columnas        : ['BusinessEntityID', 'PersonType', 'NameStyle', 'Title', 'FirstName', 'MiddleName', 'LastName', 'Suffix', 'EmailPromotion', 'ModifiedDate']


,BusinessEntityID,PersonType,NameStyle,Title,FirstName,MiddleName,LastName,Suffix,EmailPromotion,ModifiedDate
0,1,EM,False,NaN,Ken,J,Sánchez,None,0,2009-01-07
1,2,EM,False,NaN,Terri,Lee,Duffy,None,1,2008-01-24
2,3,EM,False,NaN,Roberto,NaN,Tamburello,None,0,2007-11-04
3,4,EM,False,NaN,Rob,NaN,Walters,None,0,2007-11-28
4,5,EM,False,Ms.,Gail,A,Erickson,None,0,2007-12-30
5,6,EM,False,Mr.,Jossef,H,Goldberg,None,0,2013-12-16
6,7,EM,False,NaN,Dylan,A,Miller,None,2,2009-02-01
7,8,EM,False,NaN,Diane,L,Margheim,None,0,2008-12-22
8,9,EM,False,NaN,Gigi,N,Matthew,None,0,2009-01-09
9,10,EM,False,NaN,Michael,NaN,Raheem,None,2,2009-04-26


## 9. Extracción con Filtros y Conteo Total

In [ ]:
# Contar el total de registros por tipo de persona
query_conteo = """
SELECT
    PersonType,
    COUNT(*) AS Total,
    CASE PersonType
        WHEN 'SC' THEN 'Store Contact'
        WHEN 'IN' THEN 'Individual Customer'
        WHEN 'SP' THEN 'Sales Person'
        WHEN 'EM' THEN 'Employee (no sales)'
        WHEN 'VC' THEN 'Vendor Contact'
        WHEN 'GC' THEN 'General Contact'
        ELSE 'Desconocido'
    END AS Descripcion
FROM Person.Person
GROUP BY PersonType
ORDER BY Total DESC;
"""

df_conteo = pd.read_sql(query_conteo, engine)
print('Distribución por tipo de persona:')
df_conteo

Distribución por tipo de persona:


,PersonType,Total,Descripcion
0,IN,18484,Individual Customer
1,SC,753,Store Contact
2,GC,289,General Contact
3,EM,273,Employee (no sales)
4,VC,156,Vendor Contact
5,SP,17,Sales Person


## 10. Extracción Completa y Guardado en CSV

In [ ]:

# Extraer todos los registros (sin XML para simplificar)
query_completa = """
SELECT
    BusinessEntityID,
    PersonType,
    Title,
    FirstName,
    MiddleName,
    LastName,
    Suffix,
    EmailPromotion,
    ModifiedDate
FROM Person.Person
ORDER BY BusinessEntityID;
"""

df_completa = pd.read_sql(query_completa, engine)

# ── Guardar en CSV ────────────────────────────────────────────────────────────
ruta_csv = 'datos/output/person_person.csv'
df_completa.to_csv(ruta_csv, index=False, encoding='utf-8-sig')
print(f'✅ CSV guardado     : {ruta_csv}')

# ── Guardar en Parquet ────────────────────────────────────────────────────────
ruta_parquet = 'datos/output/person_person.parquet'
df_completa.to_parquet(ruta_parquet, index=False, engine='pyarrow')
print(f'✅ Parquet guardado : {ruta_parquet}')

print(f'\nTotal de registros extraídos: {len(df_completa):,}')
df_completa.head()


✅ CSV guardado     : datos/person_person.csv
✅ Parquet guardado : datos/person_person.parquet

Total de registros extraídos: 19,972


,BusinessEntityID,PersonType,Title,FirstName,MiddleName,LastName,Suffix,EmailPromotion,ModifiedDate
0,1,EM,NaN,Ken,J,Sánchez,NaN,0,2009-01-07
1,2,EM,NaN,Terri,Lee,Duffy,NaN,1,2008-01-24
2,3,EM,NaN,Roberto,NaN,Tamburello,NaN,0,2007-11-04
3,4,EM,NaN,Rob,NaN,Walters,NaN,0,2007-11-28
4,5,EM,Ms.,Gail,A,Erickson,NaN,0,2007-12-30


## 11. Análisis Rápido del DataFrame

In [6]:
# Información general del DataFrame
print('=== Info del DataFrame ===')
df_completa.info()

print('\n=== Estadísticas descriptivas ===')
df_completa.describe(include='all')

=== Info del DataFrame ===
<class 'pandas.DataFrame'>
RangeIndex: 19972 entries, 0 to 19971
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   BusinessEntityID  19972 non-null  int64         
 1   PersonType        19972 non-null  str           
 2   Title             1009 non-null   str           
 3   FirstName         19972 non-null  str           
 4   MiddleName        11473 non-null  str           
 5   LastName          19972 non-null  str           
 6   Suffix            53 non-null     str           
 7   EmailPromotion    19972 non-null  int64         
 8   ModifiedDate      19972 non-null  datetime64[us]
dtypes: datetime64[us](1), int64(2), str(6)
memory usage: 1.7 MB

=== Estadísticas descriptivas ===


,BusinessEntityID,PersonType,Title,FirstName,MiddleName,LastName,Suffix,EmailPromotion,ModifiedDate
count,19972.000000,19972,1009,19972,11473,19972,53,19972.000000,19972
unique,NaN,6,6,1018,71,1206,6,NaN,NaN
top,NaN,IN,Mr.,Richard,A,Diaz,Jr.,NaN,NaN
freq,NaN,18484,577,103,1319,211,33,NaN,NaN
mean,10763.079411,NaN,NaN,NaN,NaN,NaN,NaN,0.630082,2013-05-18 10:25:58.388290
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,2006-06-23 00:00:00
25%,5798.750000,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,2012-12-12 00:00:00
50%,10791.500000,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,2013-09-13 00:00:00
75%,15784.250000,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,2014-01-30 00:00:00
max,20777.000000,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,2015-04-15 16:33:33.123000


In [7]:
# Valores nulos por columna
print('=== Valores nulos por columna ===')
nulos = df_completa.isnull().sum().reset_index()
nulos.columns = ['Columna', 'Valores_Nulos']
nulos['Porcentaje_%'] = (nulos['Valores_Nulos'] / len(df_completa) * 100).round(2)
nulos

=== Valores nulos por columna ===


,Columna,Valores_Nulos,Porcentaje_%
0,BusinessEntityID,0,0.00
1,PersonType,0,0.00
2,Title,18963,94.95
3,FirstName,0,0.00
4,MiddleName,8499,42.55
5,LastName,0,0.00
6,Suffix,19919,99.73
7,EmailPromotion,0,0.00
8,ModifiedDate,0,0.00


## 12. Cerrar Conexiones

In [8]:
# Cerrar la conexión directa de pyodbc
if conn:
    conn.close()
    print('Conexión pyodbc cerrada.')

# Disponer el engine de SQLAlchemy
engine.dispose()
print('Engine SQLAlchemy cerrado.')

Conexión pyodbc cerrada.
Engine SQLAlchemy cerrado.
